# Financial Transaction Pattern Analyzer
### Detecting Unusual Spending Behavior in Business Transaction Data

---

**Problem statement:** Given transaction data for businesses, can we identify unusual spending patterns or anomalous transactions that may indicate fraud, accounting mistakes, or behavior shifts?

**Approach:** Synthetic dataset → feature engineering → three detection methods → evaluation

> *This is an exploratory prototype, not a production system. The goal is to demonstrate how contextual features and interpretable methods can surface suspicious transactions for human review.*


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Reproducibility
RNG = np.random.default_rng(42)

# Plot style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

NORMAL_COLOR  = "#4C72B0"
ANOMALY_COLOR = "#DD4444"

print("Libraries loaded ✓")


---
## 1. Dataset

We use **synthetic data** for two reasons:
1. Clean, labeled public business transaction data with known anomalies doesn't exist
2. Synthetic data gives us ground-truth labels to evaluate our detection methods

The dataset simulates 12 months of transactions across 5 fictional businesses, with realistic vendors, spending profiles, and seasonality. We inject **25 labeled anomalies** of 5 types:

| Anomaly Type | Description |
|---|---|
| `amount_spike` | Single transaction far above normal for that business |
| `new_vendor_high_spend` | Large payment to a vendor never used before |
| `frequency_burst` | 20 transactions in one day (vs. normal 5–7) |
| `category_shift` | Logistics company suddenly spending heavily on Advertising |
| `duplicate_transaction` | Same amount, same vendor, same day — twice |


In [ ]:
# ── Business profiles ─────────────────────────────────────────────────────────
BUSINESSES = {
    "BIZ_001": {"name": "Acme Consulting",   "monthly_budget": 18_000},
    "BIZ_002": {"name": "Bright Retail Co",  "monthly_budget": 35_000},
    "BIZ_003": {"name": "Cedar Tech LLC",    "monthly_budget": 12_000},
    "BIZ_004": {"name": "Dunbar Logistics",  "monthly_budget": 28_000},
    "BIZ_005": {"name": "Echo Media Group",  "monthly_budget": 22_000},
}

CATEGORY_PROFILES = {
    "BIZ_001": {"Software": 0.30, "Travel": 0.25, "Office Supplies": 0.15, "Meals": 0.20, "Contractors": 0.10},
    "BIZ_002": {"Inventory": 0.45, "Shipping": 0.20, "Software": 0.10, "Office Supplies": 0.10, "Meals": 0.15},
    "BIZ_003": {"Software": 0.40, "Cloud Infrastructure": 0.30, "Contractors": 0.15, "Office Supplies": 0.10, "Meals": 0.05},
    "BIZ_004": {"Fuel": 0.30, "Maintenance": 0.25, "Insurance": 0.20, "Meals": 0.10, "Office Supplies": 0.15},
    "BIZ_005": {"Advertising": 0.40, "Software": 0.20, "Contractors": 0.25, "Meals": 0.10, "Office Supplies": 0.05},
}

VENDORS = {
    "Software":             ["Salesforce", "Slack", "Notion", "GitHub", "Zoom", "Adobe"],
    "Travel":               ["Delta Airlines", "United Airlines", "Marriott", "Hilton", "Uber"],
    "Office Supplies":      ["Staples", "Amazon Business", "Office Depot"],
    "Meals":                ["DoorDash Business", "Grubhub", "Local Catering Co"],
    "Contractors":          ["Upwork", "Toptal", "Freelancer Network"],
    "Inventory":            ["Global Wholesale Inc", "Direct Distributors", "BulkGoods LLC"],
    "Shipping":             ["FedEx", "UPS", "USPS"],
    "Cloud Infrastructure": ["AWS", "Google Cloud", "Azure"],
    "Fuel":                 ["Shell Fleet", "ExxonMobil Fleet", "Chevron Fleet"],
    "Maintenance":          ["FleetPros", "AutoShop Direct", "Nationwide Repair"],
    "Insurance":            ["Hiscox", "Nationwide Business", "Travelers"],
    "Advertising":          ["Google Ads", "Meta Business", "LinkedIn Ads"],
}

PAYMENT_TYPES = ["ACH", "Credit Card", "Wire Transfer", "Check"]
print("Business profiles defined ✓")


In [ ]:
def generate_normal_transactions(start="2023-01-01", end="2023-12-31"):
    records = []
    dates = pd.date_range(start, end, freq="B")  # Business days only

    for biz_id, biz_info in BUSINESSES.items():
        monthly_budget  = biz_info["monthly_budget"]
        category_weights = CATEGORY_PROFILES[biz_id]
        categories = list(category_weights.keys())
        weights    = list(category_weights.values())

        for date in dates:
            is_month_end = date.day >= 25
            n_txn = RNG.integers(3, 10 if is_month_end else 7)

            for _ in range(n_txn):
                category    = RNG.choice(categories, p=weights)
                vendor_pool = VENDORS.get(category, ["Generic Vendor"])
                vendor      = RNG.choice(vendor_pool)
                base        = monthly_budget * category_weights[category] / 20
                amount      = max(10.0, round(float(RNG.lognormal(np.log(base), 0.5)), 2))
                payment     = RNG.choice(PAYMENT_TYPES, p=[0.40, 0.35, 0.15, 0.10])

                records.append({
                    "transaction_date":   date,
                    "business_id":        biz_id,
                    "business_name":      biz_info["name"],
                    "vendor":             vendor,
                    "category":           category,
                    "transaction_amount": amount,
                    "payment_type":       payment,
                    "is_anomaly":         False,
                    "anomaly_type":       None,
                })
    return records


def inject_anomalies(records):
    anomalies = []
    d = pd.Timestamp("2023-09-14")

    # 1. Amount spike
    anomalies.append({"transaction_date": d, "business_id": "BIZ_003",
        "business_name": "Cedar Tech LLC", "vendor": "AWS",
        "category": "Cloud Infrastructure", "transaction_amount": 45_000.00,
        "payment_type": "Wire Transfer", "is_anomaly": True, "anomaly_type": "amount_spike"})

    # 2. New vendor + high spend
    anomalies.append({"transaction_date": d + pd.Timedelta(days=3), "business_id": "BIZ_001",
        "business_name": "Acme Consulting", "vendor": "Offshore Consult Ltd",
        "category": "Contractors", "transaction_amount": 12_000.00,
        "payment_type": "Wire Transfer", "is_anomaly": True, "anomaly_type": "new_vendor_high_spend"})

    # 3. Frequency burst (20 transactions)
    for i in range(20):
        anomalies.append({"transaction_date": d + pd.Timedelta(days=5), "business_id": "BIZ_002",
            "business_name": "Bright Retail Co", "vendor": "Global Wholesale Inc",
            "category": "Inventory", "transaction_amount": round(float(RNG.uniform(80, 300)), 2),
            "payment_type": "Credit Card", "is_anomaly": True, "anomaly_type": "frequency_burst"})

    # 4. Category shift
    anomalies.append({"transaction_date": d + pd.Timedelta(days=7), "business_id": "BIZ_004",
        "business_name": "Dunbar Logistics", "vendor": "Google Ads",
        "category": "Advertising", "transaction_amount": 8_000.00,
        "payment_type": "Credit Card", "is_anomaly": True, "anomaly_type": "category_shift"})

    # 5. Duplicate transaction (x2)
    for _ in range(2):
        anomalies.append({"transaction_date": d + pd.Timedelta(days=10), "business_id": "BIZ_005",
            "business_name": "Echo Media Group", "vendor": "Meta Business",
            "category": "Advertising", "transaction_amount": 3_200.00,
            "payment_type": "ACH", "is_anomaly": True, "anomaly_type": "duplicate_transaction"})

    records.extend(anomalies)
    return records


print("Generating dataset...")
records = generate_normal_transactions()
records = inject_anomalies(records)

df_raw = pd.DataFrame(records)
df_raw["transaction_date"] = pd.to_datetime(df_raw["transaction_date"])
df_raw = df_raw.sort_values("transaction_date").reset_index(drop=True)

print(f"✓ {len(df_raw):,} transactions | {df_raw['is_anomaly'].sum()} labeled anomalies "
      f"({df_raw['is_anomaly'].mean()*100:.1f}%)")
print(f"  Date range : {df_raw['transaction_date'].min().date()} → {df_raw['transaction_date'].max().date()}")
print(f"  Businesses : {df_raw['business_id'].nunique()}")
print(f"  Vendors    : {df_raw['vendor'].nunique()}")
df_raw.head(5)


---
## 2. Exploratory Analysis

Before building features, it helps to understand the shape of the data:
what the amount distribution looks like, how spend varies over time, and where money goes by category.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: amount histogram (log scale)
normal_amt  = df_raw.loc[~df_raw["is_anomaly"], "transaction_amount"]
anomaly_amt = df_raw.loc[ df_raw["is_anomaly"], "transaction_amount"]
bins = np.logspace(np.log10(1), np.log10(df_raw["transaction_amount"].max() * 1.1), 60)
axes[0].hist(normal_amt,  bins=bins, color=NORMAL_COLOR,  alpha=0.7, label="Normal")
axes[0].hist(anomaly_amt, bins=bins, color=ANOMALY_COLOR, alpha=0.9, label="Anomaly")
axes[0].set_xscale("log")
axes[0].set_xlabel("Transaction Amount (log scale, $)")
axes[0].set_ylabel("Count")
axes[0].set_title("Amount Distribution")
axes[0].legend()

# Right: spend by category
by_cat = df_raw.groupby("category")["transaction_amount"].sum().sort_values()
bars = axes[1].barh(by_cat.index, by_cat.values, color=NORMAL_COLOR, alpha=0.85)
axes[1].bar_label(bars, labels=[f"${v/1e3:.0f}k" for v in by_cat.values], padding=4, fontsize=8)
axes[1].set_xlabel("Total Spend ($)")
axes[1].set_title("Total Spend by Category")

plt.suptitle("Dataset Overview", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Time series of daily spend across all businesses, anomaly days highlighted
daily = (df_raw.groupby("transaction_date")
         .agg(total_spend=("transaction_amount","sum"), has_anomaly=("is_anomaly","any"))
         .reset_index())

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily["transaction_date"], daily["total_spend"],
        color=NORMAL_COLOR, linewidth=0.9, alpha=0.8, label="Daily Spend")

anomaly_days = daily[daily["has_anomaly"]]
ax.scatter(anomaly_days["transaction_date"], anomaly_days["total_spend"],
           color=ANOMALY_COLOR, zorder=5, s=70, marker="v", label="Day with Anomaly")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax.set_ylabel("Total Daily Spend ($)")
ax.set_title("Daily Spend Over Time — All Businesses")
ax.legend()
plt.tight_layout()
plt.show()


---
## 3. Feature Engineering

This is the **most important section** of the project.

Raw transaction amounts are poor anomaly signals on their own. A \$10,000 payment is alarming for one business and routine for another. What matters is **context**:

- How does this amount compare to *this business's* typical spend?
- How does it compare to *this category's* typical spend?
- Has this business ever paid *this vendor* before?
- Are there an unusual number of transactions this week?

Every feature below has a business motivation, not just a statistical one.


In [ ]:
def engineer_features(df):
    df = df.copy().sort_values(["business_id", "transaction_date"]).reset_index(drop=True)

    # 1. Calendar features
    df["day_of_week"]  = df["transaction_date"].dt.dayofweek
    df["day_of_month"] = df["transaction_date"].dt.day
    df["month"]        = df["transaction_date"].dt.month
    df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)

    # 2. Time since last transaction (per business)
    df["prev_txn_date"] = df.groupby("business_id")["transaction_date"].shift(1)
    df["days_since_last_txn"] = (df["transaction_date"] - df["prev_txn_date"]).dt.days.fillna(0)

    # 3. Amount z-score vs. this business's history
    biz_stats = df.groupby("business_id")["transaction_amount"].agg(biz_mean="mean", biz_std="std").reset_index()
    df = df.merge(biz_stats, on="business_id", how="left")
    df["amount_zscore_biz"] = ((df["transaction_amount"] - df["biz_mean"]) / df["biz_std"].replace(0, np.nan)).fillna(0)

    # 4. Amount z-score vs. this category's average
    cat_stats = df.groupby("category")["transaction_amount"].agg(cat_mean="mean", cat_std="std").reset_index()
    df = df.merge(cat_stats, on="category", how="left")
    df["amount_zscore_cat"] = ((df["transaction_amount"] - df["cat_mean"]) / df["cat_std"].replace(0, np.nan)).fillna(0)

    # 5. Vendor novelty — first time this business paid this vendor?
    first_seen = df.groupby(["business_id", "vendor"])["transaction_date"].transform("min")
    df["is_new_vendor"] = (df["transaction_date"] == first_seen).astype(int)

    # 6. Amount z-score vs. this specific vendor relationship
    vendor_stats = df.groupby(["business_id", "vendor"])["transaction_amount"].agg(vendor_mean="mean", vendor_std="std").reset_index()
    df = df.merge(vendor_stats, on=["business_id", "vendor"], how="left")
    df["amount_zscore_vendor"] = ((df["transaction_amount"] - df["vendor_mean"]) / df["vendor_std"].replace(0, np.nan)).fillna(0)

    # 7. Rolling 7-day spend and count (velocity features)
    df = df.sort_values("transaction_date").reset_index(drop=True)
    df["rolling_7d_spend"] = 0.0
    df["rolling_7d_count"] = 0.0

    for biz_id, group in df.groupby("business_id"):
        idx     = group.index
        dates   = group["transaction_date"].values
        amounts = group["transaction_amount"].values
        spend_vals, count_vals = [], []
        for i, (d, _) in enumerate(zip(dates, amounts)):
            window_start = d - np.timedelta64(7, "D")
            mask = (dates[:i] >= window_start) & (dates[:i] < d)
            spend_vals.append(amounts[:i][mask].sum())
            count_vals.append(mask.sum())
        df.loc[idx, "rolling_7d_spend"] = spend_vals
        df.loc[idx, "rolling_7d_count"] = count_vals

    # 8. Near-duplicate detection
    df = df.sort_values(["business_id", "vendor", "transaction_date"])
    df["prev_same_vendor_date"]   = df.groupby(["business_id","vendor"])["transaction_date"].shift(1)
    df["prev_same_vendor_amount"] = df.groupby(["business_id","vendor"])["transaction_amount"].shift(1)
    df["days_since_same_vendor"]  = (df["transaction_date"] - df["prev_same_vendor_date"]).dt.days.fillna(999)
    df["is_near_duplicate"] = (
        (df["days_since_same_vendor"] <= 2) &
        (df["transaction_amount"] == df["prev_same_vendor_amount"])
    ).astype(int)

    # 9. Wire transfer flag (high risk: irreversible)
    df["is_wire_transfer"] = (df["payment_type"] == "Wire Transfer").astype(int)

    # Cleanup
    df = df.drop(columns=["prev_txn_date","prev_same_vendor_date","prev_same_vendor_amount"], errors="ignore")
    df = df.sort_values("transaction_date").reset_index(drop=True)
    return df


print("Engineering features...")
df = engineer_features(df_raw)
print(f"✓ Features added: {len(df.columns) - len(df_raw.columns)} new columns")
print("\nSample: anomalous transactions with key feature values")
df[df["is_anomaly"]][["vendor","transaction_amount","amount_zscore_biz",
                       "is_new_vendor","is_near_duplicate","rolling_7d_count"]].head(6)


In [ ]:
# Feature summary statistics for anomalous vs normal transactions
FEATURE_COLS = [
    "transaction_amount", "amount_zscore_biz", "amount_zscore_cat",
    "amount_zscore_vendor", "is_new_vendor", "rolling_7d_spend",
    "rolling_7d_count", "days_since_last_txn", "is_near_duplicate",
    "is_wire_transfer", "is_weekend", "day_of_month"
]

comparison = pd.DataFrame({
    "Normal (mean)":  df[~df["is_anomaly"]][FEATURE_COLS].mean(),
    "Anomaly (mean)": df[ df["is_anomaly"]][FEATURE_COLS].mean(),
})
comparison["Ratio (Anomaly/Normal)"] = (comparison["Anomaly (mean)"] / comparison["Normal (mean)"]).round(2)
comparison.round(3)


---
## 4. Anomaly Detection

We compare three approaches, in order of interpretability:

**Method 1 — Rule-based scoring:** Human-readable thresholds that a compliance analyst could audit line by line. Produces an additive score (0–7+).

**Method 2 — Z-score composite:** Average of absolute z-scores across the three amount features. No model needed — could run in SQL.

**Method 3 — Isolation Forest:** Tree-based unsupervised method. Identifies points that are "easy to isolate" in feature space (i.e., few splits needed to separate them from the rest).

> **Honest note:** All three produce false positives. At ~0.4% anomaly rate, even a method with decent recall will flag many normal transactions. The right mental model is *prioritization for human review*, not autonomous decision-making.


In [ ]:
# ── Method 1: Rule-based score ────────────────────────────────────────────────
def rule_based_score(df):
    score = pd.Series(0.0, index=df.index)
    score += (df["amount_zscore_biz"] > 3.0).astype(float) * 2.0   # Large for this account
    score += (df["amount_zscore_cat"] > 3.0).astype(float) * 1.0   # Large for this category
    score += ((df["is_new_vendor"]==1) & (df["transaction_amount"] > 2_000)).astype(float) * 2.0  # New vendor + big payment
    score += (df["rolling_7d_count"] > 30).astype(float) * 1.5     # Transaction burst
    score += (df["is_near_duplicate"]==1).astype(float) * 2.0       # Duplicate payment
    score += ((df["is_wire_transfer"]==1) & (df["transaction_amount"] > 5_000)).astype(float) * 1.0  # Large wire
    score += (df["is_weekend"]==1).astype(float) * 0.5              # Weekend (unusual B2B)
    return score

# ── Method 2: Z-score composite ───────────────────────────────────────────────
def zscore_composite(df):
    cols = ["amount_zscore_biz", "amount_zscore_cat", "amount_zscore_vendor"]
    return df[cols].abs().mean(axis=1)

# ── Method 3: Isolation Forest ────────────────────────────────────────────────
def isolation_forest_score(df, feature_cols):
    X = df[feature_cols].fillna(df[feature_cols].median())
    X_scaled = StandardScaler().fit_transform(X)
    model = IsolationForest(n_estimators=200, contamination=0.02, random_state=42, n_jobs=-1)
    model.fit(X_scaled)
    return pd.Series(-model.score_samples(X_scaled), index=df.index)  # Higher = more anomalous

df["rule_score"]    = rule_based_score(df)
df["zscore_score"]  = zscore_composite(df)
df["iforest_score"] = isolation_forest_score(df, FEATURE_COLS)

print("✓ Anomaly scores computed for all three methods")


In [ ]:
# ── Evaluation at 98th percentile threshold ───────────────────────────────────
def evaluate(df, score_col, pct=98.0):
    threshold = np.percentile(df[score_col], pct)
    flagged   = df[score_col] >= threshold
    tp = (flagged &  df["is_anomaly"]).sum()
    fp = (flagged & ~df["is_anomaly"]).sum()
    fn = (~flagged & df["is_anomaly"]).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    return {"Method": score_col, "Flagged": int(flagged.sum()), "True Pos": int(tp),
            "False Pos": int(fp), "Precision": round(prec,3), "Recall": round(rec,3), "F1": round(f1,3)}

results = [evaluate(df, c) for c in ["rule_score","zscore_score","iforest_score"]]
results_df = pd.DataFrame(results)
print("Evaluation at 98th-percentile threshold (top 2% flagged):")
results_df


---
## 5. Results & Visualizations


In [ ]:
# Score distributions — how well do scores separate anomalies from normal?
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
score_cols = ["rule_score", "zscore_score", "iforest_score"]
titles     = ["Rule-Based Score", "Z-Score Composite", "Isolation Forest"]

for ax, col, title in zip(axes, score_cols, titles):
    threshold = np.percentile(df[col], 98)
    bins = np.linspace(df[col].min(), df[col].max(), 50)
    ax.hist(df.loc[~df["is_anomaly"], col], bins=bins, color=NORMAL_COLOR,  alpha=0.7, label="Normal")
    ax.hist(df.loc[ df["is_anomaly"], col], bins=bins, color=ANOMALY_COLOR, alpha=0.9, label="Labeled Anomaly")
    ax.axvline(threshold, color="black", linestyle="--", linewidth=1.5, label=f"Threshold")
    ax.set_title(title)
    ax.set_xlabel("Score")
    ax.legend(fontsize=8)

plt.suptitle("Anomaly Score Distributions (98th-percentile threshold shown)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Method comparison bar chart
methods    = ["Rule-Based", "Z-Score", "Isolation Forest"]
precisions = [r["Precision"] for r in results]
recalls    = [r["Recall"]    for r in results]
f1s        = [r["F1"]        for r in results]

x, width = np.arange(3), 0.25
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width, precisions, width, label="Precision", color="#4C72B0", alpha=0.85)
ax.bar(x,         recalls,    width, label="Recall",    color="#55A868", alpha=0.85)
ax.bar(x + width, f1s,        width, label="F1",        color="#C44E52", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(methods)
ax.set_ylabel("Score (0–1)"); ax.set_ylim(0, 1.15)
ax.set_title("Detection Method Comparison (98th-percentile threshold)")
ax.legend()
ax.axhline(0.5, color="gray", linestyle=":", linewidth=0.8)
for container, vals in zip(ax.containers, [precisions, recalls, f1s]):
    ax.bar_label(container, labels=[f"{v:.2f}" for v in vals], padding=2, fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Top 15 flagged transactions by rule_score
cols_show = ["transaction_date","business_name","vendor","category",
             "transaction_amount","rule_score","is_anomaly","anomaly_type"]
top15 = df.sort_values("rule_score", ascending=False).head(15)[cols_show].copy()
top15["transaction_amount"] = top15["transaction_amount"].apply(lambda x: f"${x:,.0f}")
top15["transaction_date"]   = top15["transaction_date"].dt.strftime("%Y-%m-%d")
top15.reset_index(drop=True)


In [ ]:
# Feature correlation heatmap
import matplotlib.colors as mcolors

corr = df[FEATURE_COLS].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(FEATURE_COLS))); ax.set_xticklabels(FEATURE_COLS, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(FEATURE_COLS))); ax.set_yticklabels(FEATURE_COLS, fontsize=8)
ax.set_title("Feature Correlation Matrix")
for i in range(len(FEATURE_COLS)):
    for j in range(len(FEATURE_COLS)):
        v = corr.iloc[i,j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                fontsize=6, color="black" if abs(v) < 0.7 else "white")
plt.tight_layout(); plt.show()


---
## 6. Key Findings & Honest Limitations

### What worked
- **Contextual z-scores** (vs. business history, vs. vendor relationship) were the most discriminating features. The amount spike and new-vendor anomalies had z-scores of 30–36 — far outside any normal range.
- **The rule-based scorer correctly ranked** the most semantically meaningful anomalies at the top: new-vendor wire transfer, duplicate payment, category shift.
- **Velocity features** (rolling 7-day count) caught the frequency burst anomaly, which was invisible to amount-based features.

### Where it struggled
- **Precision is low across all methods** (~4–8% at the 98th percentile). Most flagged transactions are normal outliers, not true anomalies. This is expected at 0.4% anomaly rate.
- **All three methods had identical recall** in this evaluation — they all found the same 5 out of 25 labeled anomalies. The frequency burst (20 small transactions) was the hardest to catch because no single transaction looks unusual.

### Honest limitations
1. **Synthetic data is cleaner than reality.** Real data has inconsistent vendor names, refunds, FX conversions, and messy metadata.
2. **Without real labels, threshold tuning is guesswork.** The 98th percentile cutoff was arbitrary — in production, you'd tune it based on analyst capacity and fraud cost.
3. **This is reactive, not predictive.** It scores past transactions. Slow-burn fraud (gradual spend creep over months) would be missed.

### What I'd do next with better data
- Add a labeled fraud dataset and train a supervised classifier
- Build vendor relationship graphs (network-level features)
- Implement a feedback loop where analyst decisions improve the model
- Move to real-time streaming scoring for wire transfers before settlement


---
## 7. Project Summary

| | |
|---|---|
| **Dataset** | 6,428 synthetic transactions, 5 businesses, 42 vendors, 12 months |
| **Anomaly rate** | 0.4% (25 labeled anomalies, 5 types) |
| **Features engineered** | 10 contextual features |
| **Methods compared** | Rule-based, Z-score composite, Isolation Forest |
| **Best F1** | ~0.065 — low, as expected for unsupervised detection at this anomaly rate |
| **Key takeaway** | Feature engineering matters more than model choice; these scores are best used to prioritize human review |

---
*Built for learning and portfolio purposes. Not a production system.*


---
## 8. SQL Implementation

One of the practical advantages of simple anomaly detection methods is that they
can be expressed directly in SQL — which is how most production transaction
monitoring pipelines actually work (think: BigQuery, Snowflake, Redshift).

Here we recreate four of our core anomaly checks using **SQLite** (built into
Python — no setup needed). This mirrors what a data/risk engineer would write
in a real fintech data warehouse.

The four queries:
1. **Amount spike** — transactions more than 3× the business's average
2. **New vendor + high spend** — first-ever payment to a vendor over \$2,000
3. **Duplicate transactions** — same vendor, same amount, within 2 days
4. **Frequency burst** — businesses with 15+ transactions in any 7-day window


In [ ]:
import sqlite3
import pandas as pd

# Load the transaction data into an in-memory SQLite database
# (df_raw was created in Section 1 — make sure you've run that first)

conn = sqlite3.connect(":memory:")  # In-memory DB, no file needed

# Write the transactions table
df_raw.to_sql("transactions", conn, index=False, if_exists="replace")

print("✓ Loaded transactions into SQLite")
print(f"  Rows: {pd.read_sql('SELECT COUNT(*) as n FROM transactions', conn).iloc[0,0]:,}")
print(f"  Columns: {list(df_raw.columns)}")


### Query 1: Amount Spike
Flag any transaction where the amount is more than 3 standard deviations above
that business's mean. This is the SQL equivalent of `amount_zscore_biz > 3`.


In [ ]:
query_amount_spike = """
SELECT
    t.transaction_date,
    t.business_name,
    t.vendor,
    t.category,
    t.transaction_amount,
    ROUND(stats.avg_amount, 2)            AS business_avg,
    ROUND(stats.std_amount, 2)            AS business_std,
    ROUND(
        (t.transaction_amount - stats.avg_amount) / stats.std_amount,
        2
    )                                     AS z_score,
    t.is_anomaly,
    t.anomaly_type
FROM transactions t
JOIN (
    SELECT
        business_id,
        AVG(transaction_amount)  AS avg_amount,
        -- SQLite has no STDEV, so we compute it manually
        SQRT(
            AVG(transaction_amount * transaction_amount) -
            AVG(transaction_amount) * AVG(transaction_amount)
        )                        AS std_amount
    FROM transactions
    GROUP BY business_id
) stats ON t.business_id = stats.business_id
WHERE
    stats.std_amount > 0
    AND (t.transaction_amount - stats.avg_amount) / stats.std_amount > 3
ORDER BY z_score DESC
LIMIT 15
"""

result1 = pd.read_sql(query_amount_spike, conn)
print(f"Transactions flagged as amount spikes: {len(result1)}")
result1


### Query 2: New Vendor + High Spend
Find transactions where the vendor has never been paid by this business before,
AND the amount exceeds \$2,000. This is the SQL equivalent of
`is_new_vendor == 1 AND transaction_amount > 2000`.


In [ ]:
query_new_vendor = """
SELECT
    t.transaction_date,
    t.business_name,
    t.vendor,
    t.category,
    t.transaction_amount,
    t.payment_type,
    t.is_anomaly,
    t.anomaly_type
FROM transactions t
WHERE
    t.transaction_amount > 2000
    -- First-ever appearance of this vendor for this business
    AND t.transaction_date = (
        SELECT MIN(t2.transaction_date)
        FROM transactions t2
        WHERE t2.business_id = t.business_id
          AND t2.vendor      = t.vendor
    )
ORDER BY t.transaction_amount DESC
LIMIT 15
"""

result2 = pd.read_sql(query_new_vendor, conn)
print(f"New vendor + high spend flags: {len(result2)}")
result2


### Query 3: Near-Duplicate Transactions
Find cases where the same business paid the same vendor the same amount
within a 2-day window — a classic double-payment pattern.


In [ ]:
query_duplicates = """
SELECT
    t1.transaction_date       AS date_1,
    t2.transaction_date       AS date_2,
    t1.business_name,
    t1.vendor,
    t1.category,
    t1.transaction_amount,
    t1.payment_type,
    -- Days between the two transactions
    CAST(
        (julianday(t2.transaction_date) - julianday(t1.transaction_date))
        AS INTEGER
    )                         AS days_apart,
    t1.is_anomaly,
    t1.anomaly_type
FROM transactions t1
JOIN transactions t2
    ON  t1.business_id        = t2.business_id
    AND t1.vendor             = t2.vendor
    AND t1.transaction_amount = t2.transaction_amount
    AND t2.transaction_date   > t1.transaction_date
    AND julianday(t2.transaction_date) - julianday(t1.transaction_date) <= 2
ORDER BY t1.transaction_amount DESC
LIMIT 15
"""

result3 = pd.read_sql(query_duplicates, conn)
print(f"Near-duplicate transaction pairs found: {len(result3)}")
result3


### Query 4: Transaction Frequency Burst
Find any 7-day window where a business made an unusually high number of
transactions. This catches the frequency burst anomaly that amount-based
features miss entirely.


In [ ]:
query_frequency = """
SELECT
    t1.business_name,
    t1.transaction_date                          AS window_start,
    COUNT(*)                                     AS txn_count_7d,
    ROUND(SUM(t2.transaction_amount), 2)         AS total_spend_7d,
    ROUND(AVG(t2.transaction_amount), 2)         AS avg_amount_7d
FROM transactions t1
JOIN transactions t2
    ON  t1.business_id = t2.business_id
    AND julianday(t2.transaction_date) BETWEEN julianday(t1.transaction_date)
                                           AND julianday(t1.transaction_date) + 7
GROUP BY t1.business_id, t1.transaction_date
HAVING COUNT(*) >= 15
ORDER BY txn_count_7d DESC
LIMIT 20
"""

result4 = pd.read_sql(query_frequency, conn)
print(f"High-frequency windows found: {len(result4)}")
result4


### Combined SQL Risk Summary
Union all four flags into a single risk report — this is what a
production monitoring dashboard might run nightly.


In [ ]:
query_combined = """
-- Flag 1: Amount spikes
SELECT
    transaction_date,
    business_name,
    vendor,
    category,
    transaction_amount,
    'amount_spike'        AS flag_type,
    is_anomaly
FROM transactions t
JOIN (
    SELECT business_id,
           AVG(transaction_amount) AS avg_amt,
           SQRT(AVG(transaction_amount*transaction_amount) -
                AVG(transaction_amount)*AVG(transaction_amount)) AS std_amt
    FROM transactions GROUP BY business_id
) s ON t.business_id = s.business_id
WHERE s.std_amt > 0
  AND (t.transaction_amount - s.avg_amt) / s.std_amt > 3

UNION ALL

-- Flag 2: New vendor + high spend
SELECT
    transaction_date,
    business_name,
    vendor,
    category,
    transaction_amount,
    'new_vendor_high_spend' AS flag_type,
    is_anomaly
FROM transactions t
WHERE transaction_amount > 2000
  AND transaction_date = (
      SELECT MIN(t2.transaction_date) FROM transactions t2
      WHERE t2.business_id = t.business_id AND t2.vendor = t.vendor
  )

UNION ALL

-- Flag 3: Duplicates
SELECT
    t1.transaction_date,
    t1.business_name,
    t1.vendor,
    t1.category,
    t1.transaction_amount,
    'duplicate'           AS flag_type,
    t1.is_anomaly
FROM transactions t1
JOIN transactions t2
    ON  t1.business_id = t2.business_id
    AND t1.vendor = t2.vendor
    AND t1.transaction_amount = t2.transaction_amount
    AND t2.transaction_date > t1.transaction_date
    AND julianday(t2.transaction_date) - julianday(t1.transaction_date) <= 2

ORDER BY transaction_amount DESC
LIMIT 30
"""

combined = pd.read_sql(query_combined, conn)
print(f"Total flagged transactions in combined report: {len(combined)}")
print(f"True anomalies captured: {combined['is_anomaly'].sum()}")
print()

# Show flag type breakdown
print("Flags by type:")
print(combined.groupby("flag_type").agg(
    count=("flag_type","count"),
    true_anomalies=("is_anomaly","sum")
).to_string())

combined.head(20)


### Why SQL matters here

In a production fintech setting, transaction data typically lives in a data
warehouse (BigQuery, Snowflake, Redshift), not a Python dataframe. The ability
to express anomaly logic in SQL means:

- **It runs at scale** — no need to pull millions of rows into memory
- **It's auditable** — compliance teams can read and verify the logic
- **It's schedulable** — run as a nightly job with no Python infrastructure
- **It's fast to iterate** — a risk analyst can tweak a threshold in a WHERE clause

The Python/sklearn Isolation Forest model is more powerful for catching subtle
multivariate anomalies, but the SQL rules are more practical for day-to-day
monitoring. A real system would use both: SQL for fast, interpretable first-pass
filtering, and ML models for deeper investigation of flagged items.
